# Kaggle Titanic ノートブック

このノートブックは1つのファイルで完結するテンプレートです。セットアップ、データ読み込み、EDA、特徴量エンジニアリング、モデル学習、提出を順に実行します。

## セットアップ

ローカルでは `uv` を使い、`pyproject.toml` に依存を記載しています。依存インストール例:
```bash
uv init
uv sync
```

## データダウンロード

以下コマンドをCLI実行することでデータをダウンロード
```bash
export KAGGLE_API_TOKEN=xxxxx
kaggle competitions download -c titanic -p data
unzip data/titanic.zip -d data/
```

In [21]:
# インポート
import os
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import LabelEncoder

# LightGBM のインポート（無ければエラーを出す）
try:
    from lightgbm import LGBMClassifier
except Exception:
    print('LightGBM が使えません。uv でインストールしてください: uv add lightgbm')
    raise

%matplotlib inline
sns.set_style('whitegrid')

In [28]:
# データ読み込み
TRAIN_PATH = Path('../data/train.csv')
TEST_PATH = Path('../data/test.csv')
if not (TRAIN_PATH.exists() and TEST_PATH.exists()):
    print('data/train.csv と data/test.csv を配置してください')
else:
    df_train = pd.read_csv(TRAIN_PATH)
    df_test = pd.read_csv(TEST_PATH)
    print('train', df_train.shape, 'test', df_test.shape)

train (891, 12) test (418, 11)


In [29]:
# EDA（簡易）
display(df_train.head())
print(df_train.info())
display(df_train.isnull().sum())

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


<class 'pandas.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  891 non-null    int64  
 1   Survived     891 non-null    int64  
 2   Pclass       891 non-null    int64  
 3   Name         891 non-null    str    
 4   Sex          891 non-null    str    
 5   Age          714 non-null    float64
 6   SibSp        891 non-null    int64  
 7   Parch        891 non-null    int64  
 8   Ticket       891 non-null    str    
 9   Fare         891 non-null    float64
 10  Cabin        204 non-null    str    
 11  Embarked     889 non-null    str    
dtypes: float64(2), int64(5), str(5)
memory usage: 83.7 KB
None


PassengerId      0
Survived         0
Pclass           0
Name             0
Sex              0
Age            177
SibSp            0
Parch            0
Ticket           0
Fare             0
Cabin          687
Embarked         2
dtype: int64

## 特徴量エンジニアリング

以下の派生特徴を作成します: `Title`, `FamilySize`, `IsAlone`, `Deck`, `CabinCount`, `TicketPrefix`, `TicketCount`, `NameLen`, `FareLog`, `Age`補完, `IsChild` 等

In [32]:
def feature_engineering(df):
    df = df.copy()
    # Title を抽出（Name から敬称）
    df['Title'] = df['Name'].str.extract(',\s*([^\.]+)\.', expand=False).str.strip()
    df['Title'] = df['Title'].replace({'Mlle':'Miss', 'Ms':'Miss', 'Mme':'Mrs'})
    rare_titles = ['Lady','Countess','Capt','Col','Don','Dr','Major','Rev','Sir','Jonkheer','Dona']
    df['Title'] = df['Title'].apply(lambda x: 'Rare' if x in rare_titles else x)
    # Family size
    df['FamilySize'] = df['SibSp'] + df['Parch'] + 1
    df['IsAlone'] = (df['FamilySize'] == 1).astype(int)
    # Cabin 系
    df['Cabin'] = df['Cabin'].fillna('Unknown')
    df['Deck'] = df['Cabin'].str[0]
    df['CabinCount'] = df['Cabin'].apply(lambda x: len(str(x).split()) if x != 'Unknown' else 0)
    # Ticket プレフィックスとチケット共有人数
    df['TicketPrefix'] = df['Ticket'].str.replace(r'[^A-Za-z]', ' ', regex=True).str.strip().str.split().str[0].fillna('NUM')
    df['TicketPrefix'] = df['TicketPrefix'].replace('', 'NUM')
    df['TicketCount'] = df.groupby('Ticket')['Ticket'].transform('count')
    # Name 関連
    df['NameLen'] = df['Name'].str.len()
    df['NameWordCount'] = df['Name'].str.split().apply(lambda x: len(x))
    # Fare の対数変換
    df['Fare'] = df['Fare'].fillna(0)
    df['FareLog'] = np.log1p(df['Fare'])
    # Age の補完（Title 母集団の中央値で簡易補完）
    df['Age'] = df['Age'].astype(float)
    age_median_by_title = df.groupby('Title')['Age'].transform('median')
    df['Age'] = df['Age'].fillna(age_median_by_title)
    df['Age'] = df['Age'].fillna(df['Age'].median())
    df['IsChild'] = (df['Age'] < 16).astype(int)
    df['AgeBin'] = pd.cut(df['Age'], bins=[0,12,18,35,60,200], labels=False)
    return df

# train/test 両方に適用（簡便のため結合して FE を実施）
# combined = pd.concat([df_train.drop(columns=['Survived']), df_test], sort=False).reset_index(drop=True)
# combined_fe = feature_engineering(combined)
train_fe = feature_engineering(df_train)
test_fe = feature_engineering(df_test)
print('特徴量エンジニアリング完了')

特徴量エンジニアリング完了


<>:4: SyntaxWarning: invalid escape sequence '\s'
<>:4: SyntaxWarning: invalid escape sequence '\s'
/tmp/ipykernel_24404/3399330863.py:4: SyntaxWarning: invalid escape sequence '\s'
  df['Title'] = df['Name'].str.extract(',\s*([^\.]+)\.', expand=False).str.strip()


### OOF ターゲットエンコーディング

高 cardinality カテゴリに対して OOF ターゲットエンコーディングを行います。必ず fold 内で計算して val に適用します。

In [33]:
def target_encode_oof(train_df, test_df, col, target='Survived', n_splits=5, seed=42, smoothing=1.0, noise_level=0.01):
    oof = pd.Series(index=train_df.index, dtype=float)
    global_mean = train_df[target].mean()
    kf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    for tr_idx, val_idx in kf.split(train_df, train_df[target]):
        agg = train_df.iloc[tr_idx].groupby(col)[target].agg(['mean', 'count'])
        smooth = (agg['mean'] * agg['count'] + global_mean * smoothing) / (agg['count'] + smoothing)
        mapping = smooth.to_dict()
        oof.iloc[val_idx] = train_df.iloc[val_idx][col].map(mapping).fillna(global_mean)
    agg_full = train_df.groupby(col)[target].agg(['mean', 'count'])
    smooth_full = (agg_full['mean'] * agg_full['count'] + global_mean * smoothing) / (agg_full['count'] + smoothing)
    test_encoded = test_df[col].map(smooth_full.to_dict()).fillna(global_mean)
    np.random.seed(seed)
    oof = oof * (1 + noise_level * np.random.randn(len(oof)))
    return oof, test_encoded

oof_te, test_te = target_encode_oof(train_fe, test_fe, 'TicketPrefix')
train_fe['TicketPrefix_te'] = oof_te
test_fe['TicketPrefix_te'] = test_te
print('OOF encoding done')

OOF encoding done


In [38]:
train_fe.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,...,CabinCount,TicketPrefix,TicketCount,NameLen,NameWordCount,FareLog,IsChild,AgeBin,TicketPrefix_te,oof_pred
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,...,0,A,1,23,4,2.110213,0,2,0.095827,0.006982
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,...,1,PC,1,51,7,4.280593,0,3,0.637896,0.999999
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,...,0,STON,1,22,3,2.188856,0,2,0.377952,0.117218
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,...,1,NUM,2,44,7,3.990834,0,2,0.396561,1.000000
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,...,0,NUM,1,24,4,2.202765,0,2,0.374601,0.001055


## モデル学習と予測

LightGBM を用いた 5-fold CV（Stratified）で OOF を計算し、テスト予測は fold 平均を取ります。

In [36]:
# 学習・予測セル
features = [
    'Pclass','Sex','FareLog','Age','FamilySize','IsAlone','Title','Deck',
    'CabinCount','TicketPrefix_te','TicketCount','NameLen','IsChild','AgeBin','Embarked'
]

# 入力データ準備
X_train = train_fe[features].copy()
X_test = test_fe[features].copy()
y = train_fe['Survived'].values

# 文字列カテゴリを整数にマップ（train の値に基づき、未知は -1）
cat_cols = X_train.select_dtypes(include=['object']).columns.tolist()
for col in cat_cols:
    mapping = {v: i for i, v in enumerate(X_train[col].fillna('NA').astype(str).unique())}
    X_train[col] = X_train[col].fillna('NA').astype(str).map(mapping).fillna(-1).astype(int)
    X_test[col] = X_test[col].fillna('NA').astype(str).map(mapping).fillna(-1).astype(int)

# AgeBin 整数化
if 'AgeBin' in X_train.columns:
    X_train['AgeBin'] = X_train['AgeBin'].astype(float).fillna(-1).astype(int)
    X_test['AgeBin'] = X_test['AgeBin'].astype(float).fillna(-1).astype(int)

# CV 学習
oof_preds = np.zeros(len(X_train))
test_preds = np.zeros(len(X_test))
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
for fold, (tr_idx, val_idx) in enumerate(skf.split(X_train, y)):
    X_tr, X_val = X_train.iloc[tr_idx], X_train.iloc[val_idx]
    y_tr, y_val = y[tr_idx], y[val_idx]
    model = LGBMClassifier(n_estimators=10000, learning_rate=0.05, random_state=42)
    model.fit(
        X_tr, y_tr,
        eval_set=[(X_val, y_val)],
        # early_stopping_rounds=100,
        eval_metric='auc',
        # verbose=100,
    )
    oof_preds[val_idx] = model.predict_proba(X_val)[:, 1]
    test_preds += model.predict_proba(X_test)[:, 1] / skf.n_splits

# 評価
cv_auc = roc_auc_score(y, oof_preds)
print(f'CV AUC: {cv_auc:.5f}')

# 結果を保存
train_fe['oof_pred'] = oof_preds
test_fe['pred_proba'] = test_preds
test_fe['Survived'] = (test_fe['pred_proba'] > 0.5).astype(int)
submission = test_fe[['PassengerId', 'Survived']].copy()
submission.to_csv('submission.csv', index=False)
print('submission.csv を作成しました（ルートに保存）。')
display(submission.head())

/tmp/ipykernel_24404/2114498914.py:13: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = X_train.select_dtypes(include=['object']).columns.tolist()


[LightGBM] [Info] Number of positive: 273, number of negative: 439
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000501 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 514
[LightGBM] [Info] Number of data points in the train set: 712, number of used features: 15
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.383427 -> initscore=-0.475028
[LightGBM] [Info] Start training from score -0.475028
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf

,PassengerId,Survived
0,892,0
1,893,0
2,894,0
3,895,0
4,896,1


In [37]:
train_fe.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,...,CabinCount,TicketPrefix,TicketCount,NameLen,NameWordCount,FareLog,IsChild,AgeBin,TicketPrefix_te,oof_pred
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,...,0,A,1,23,4,2.110213,0,2,0.095827,0.006982
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,...,1,PC,1,51,7,4.280593,0,3,0.637896,0.999999
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,...,0,STON,1,22,3,2.188856,0,2,0.377952,0.117218
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,...,1,NUM,2,44,7,3.990834,0,2,0.396561,1.000000
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,...,0,NUM,1,24,4,2.202765,0,2,0.374601,0.001055
